# Job API Scraper

List all non-deleted jobs in a Databricks workspace and return a pandas DataFrame.

Run this in a notebook within the workspace you want to query.
The SDK auto-authenticates using the notebook's context.

**Output columns:**
- `job_id`, `job_name`, `job_tags`, `task_key`, `task_type`, `cluster_type`,
  `notebook_path`, `notebook_name`, `notebook_language`,
  `last_run_duration_seconds`, `last_run_status`, `last_run_end_time`

In [0]:
import pandas as pd
from databricks.sdk import WorkspaceClient

# ── Configuration ──────────────────────────────────────────────────────────────
CATALOG = "prod_catalog"
SCHEMA = "databricks_serverless_db"
SOURCE_TABLE = "workspace_jobs"
SUMMARY_TABLE = "workspace_jobs_summary"

full_source = f"{CATALOG}.{SCHEMA}.{SOURCE_TABLE}"
full_summary = f"{CATALOG}.{SCHEMA}.{SUMMARY_TABLE}"

In [0]:
# Task types that are control flow only — no compute involved
_NO_COMPUTE_TASK_TYPES = {"condition", "for_each", "run_job"}


def _get_cluster_type(task, task_type: str) -> str:
    """Determine cluster type from task compute configuration."""
    if task_type in _NO_COMPUTE_TASK_TYPES:
        return "none"
    if task.job_cluster_key:
        return "job_cluster"
    if task.existing_cluster_id:
        return "interactive"
    if task.new_cluster:
        return "job_cluster"
    if getattr(task, "environment_key", None):
        return "serverless"
    if task.notebook_task or task.spark_python_task or task.sql_task:
        return "serverless"
    if task_type == "clean_rooms_notebook":
        return "clean_rooms"
    if task_type == "gen_ai_compute":
        return "gen_ai_compute"
    return "unknown"


def _get_last_run_info(client: WorkspaceClient, job_id: int) -> dict:
    """Get the most recent run's duration, status, and end time."""
    info = {
        "last_run_duration_seconds": None,
        "last_run_status": None,
        "last_run_end_time": None,
    }
    try:
        run = next(iter(client.jobs.list_runs(job_id=job_id, limit=1)), None)
        if run:
            if run.end_time and run.start_time:
                info["last_run_duration_seconds"] = round(
                    (run.end_time - run.start_time) / 1000, 1
                )
            if run.end_time:
                info["last_run_end_time"] = pd.Timestamp(
                    run.end_time, unit="ms", tz="UTC"
                )
            if run.state and run.state.result_state:
                info["last_run_status"] = run.state.result_state.value
            elif run.state and run.state.life_cycle_state:
                info["last_run_status"] = run.state.life_cycle_state.value
    except Exception:
        pass
    return info


def collect_jobs(client: WorkspaceClient) -> pd.DataFrame:
    """Iterate over all active jobs and extract task/notebook info."""
    rows = []

    print("Fetching job list...")
    all_jobs = list(client.jobs.list(expand_tasks=True))
    total_jobs = len(all_jobs)
    print(f"Found {total_jobs} jobs. Collecting details...")

    for i, job in enumerate(all_jobs, 1):
        if i % 25 == 0 or i == total_jobs:
            print(f"  Processing job {i}/{total_jobs} ({100 * i // total_jobs}%)")

        job_id = job.job_id
        job_name = job.settings.name if job.settings else None

        tags = {}
        if job.settings and job.settings.tags:
            tags = job.settings.tags
        job_tags = dict(tags) if tags else None

        run_info = _get_last_run_info(client, job_id)

        tasks = []
        if job.settings and job.settings.tasks:
            tasks = job.settings.tasks

        if not tasks:
            rows.append({
                "job_id": job_id,
                "job_name": job_name,
                "job_tags": job_tags,
                "task_key": None,
                "task_type": None,
                "cluster_type": None,
                "notebook_path": None,
                "notebook_name": None,
                "notebook_language": None,
                **run_info,
            })
            continue

        for task in tasks:
            task_key = task.task_key
            notebook_path = None
            notebook_name = None
            notebook_language = None
            task_type = "other"

            if task.notebook_task:
                task_type = "notebook"
                notebook_path = task.notebook_task.notebook_path
                if notebook_path:
                    notebook_name = notebook_path.rsplit("/", 1)[-1]
                    try:
                        obj_info = client.workspace.get_status(notebook_path)
                        notebook_language = (
                            obj_info.language.value if obj_info.language else None
                        )
                    except Exception:
                        notebook_language = None
            elif task.spark_python_task:
                task_type = "spark_python"
            elif task.python_wheel_task:
                task_type = "python_wheel"
            elif task.spark_jar_task:
                task_type = "spark_jar"
            elif task.spark_submit_task:
                task_type = "spark_submit"
            elif task.pipeline_task:
                task_type = "pipeline"
            elif task.sql_task:
                task_type = "sql"
            elif task.dbt_task:
                task_type = "dbt"
            elif task.run_job_task:
                task_type = "run_job"
            elif getattr(task, "condition_task", None):
                task_type = "condition"
            elif getattr(task, "for_each_task", None):
                task_type = "for_each"
            elif getattr(task, "clean_rooms_notebook_task", None):
                task_type = "clean_rooms_notebook"
            elif getattr(task, "gen_ai_compute_task", None):
                task_type = "gen_ai_compute"

            cluster_type = _get_cluster_type(task, task_type)

            rows.append({
                "job_id": job_id,
                "job_name": job_name,
                "job_tags": job_tags,
                "task_key": task_key,
                "task_type": task_type,
                "cluster_type": cluster_type,
                "notebook_path": notebook_path,
                "notebook_name": notebook_name,
                "notebook_language": notebook_language,
                **run_info,
            })

    df = pd.DataFrame(rows, columns=[
        "job_id", "job_name", "job_tags", "task_key", "task_type", "cluster_type",
        "notebook_path", "notebook_name", "notebook_language",
        "last_run_duration_seconds", "last_run_status", "last_run_end_time",
    ])
    print(f"Done. Collected {len(df)} task rows from {df['job_id'].nunique()} jobs.")
    return df

## Write results to Delta table

In [0]:
client = WorkspaceClient()
df = collect_jobs(client)

# ── Convert job_tags dict to string for Delta compatibility ────────────────────
df["job_tags"] = df["job_tags"].apply(lambda x: str(x) if x else None)

# ── Write to Delta table ──────────────────────────────────────────────────────
print(f"Writing {len(df)} rows to {full_source}...")

spark_df = spark.createDataFrame(df)
spark_df.write.format("delta").mode("overwrite").saveAsTable(full_source)

print(f"Successfully wrote to {full_source}")

## Summarize to job-level table

In [ ]:
from datetime import datetime, timezone
from collections import Counter

df = spark.read.table(full_source).toPandas()

# ── Define pivot categories ───────────────────────────────────────────────────
TASK_TYPES = [
    "notebook", "spark_python", "python_wheel", "spark_jar", "spark_submit",
    "pipeline", "sql", "dbt", "run_job", "condition", "for_each",
    "clean_rooms_notebook", "gen_ai_compute", "other",
]
LANGUAGES = ["PYTHON", "SQL", "SCALA", "R"]
COMPUTE_TYPES = [
    "job_cluster", "interactive", "serverless", "none",
    "clean_rooms", "gen_ai_compute", "unknown",
]

# Task type to language mapping
_TASK_LANGUAGE_MAP = {
    "spark_python": "python",
    "python_wheel": "python",
    "sql": "sql",
    "spark_jar": "investigation_needed",
    "other": "investigation_needed",
}
# These task types are skipped (no language contribution)
_SKIP_TASK_TYPES = {
    "spark_submit", "pipeline", "dbt", "run_job",
    "condition", "for_each", "clean_rooms_notebook", "gen_ai_compute",
}

MOSTLY_THRESHOLD = 0.8

six_months_ago = pd.Timestamp(datetime.now(timezone.utc)).tz_localize(None) - pd.DateOffset(months=6)


def _get_task_language(task_type, notebook_language):
    """Map a single task row to its language contribution."""
    if task_type in _SKIP_TASK_TYPES:
        return None
    if task_type == "notebook":
        if notebook_language and isinstance(notebook_language, str):
            return notebook_language.lower()
        return None  # couldn't determine — skip rather than pollute
    return _TASK_LANGUAGE_MAP.get(task_type)


def _derive_job_language(languages):
    """Derive a single job_language label from a list of per-task languages."""
    # Filter out Nones (skipped/undetermined tasks)
    langs = [l for l in languages if l is not None and l != "investigation_needed"]
    has_investigation = any(l == "investigation_needed" for l in languages if l is not None)
    suffix = "_needs_investigation" if has_investigation else ""

    if not langs:
        return "no_language_tasks" + suffix

    counts = Counter(langs)
    total = sum(counts.values())
    unique = sorted(counts.keys())

    if len(unique) == 1:
        return unique[0] + suffix

    # Check for "mostly" one language (>=80%)
    top_lang, top_count = counts.most_common(1)[0]
    if top_count / total >= MOSTLY_THRESHOLD:
        return f"mostly_{top_lang}" + suffix

    # Two languages
    if len(unique) == 2:
        return f"{unique[0]}_and_{unique[1]}" + suffix

    # 3+ languages
    return "mixed" + suffix


# ── Build summary per job ─────────────────────────────────────────────────────
rows = []
for (job_id, job_name, job_tags), group in df.groupby(
    ["job_id", "job_name", "job_tags"], dropna=False
):
    # Use first row for job-level fields (same across all tasks in a job)
    last_run_end_time = group["last_run_end_time"].dropna().iloc[0] if group["last_run_end_time"].notna().any() else None

    # Derive per-task languages
    task_languages = [
        _get_task_language(r["task_type"], r.get("notebook_language"))
        for _, r in group.iterrows()
    ]
    has_investigation = any(
        r["task_type"] in ("spark_jar", "other")
        for _, r in group.iterrows()
    )

    row = {
        "job_id": job_id,
        "job_name": job_name,
        "job_tags": job_tags,
        "num_tasks": group["task_key"].nunique() if group["task_key"].notna().any() else 0,
        "last_run_end_time": last_run_end_time,
        "ran_last_6_months": "Y" if (last_run_end_time is not None and last_run_end_time >= six_months_ago) else "N",
        "job_language": _derive_job_language(task_languages),
        "investigation_needed": "Y" if has_investigation else "N",
    }

    # Task type counts — fillna so nulls don't wipe out counts
    task_counts = group["task_type"].fillna("other").value_counts()
    for tt in TASK_TYPES:
        row[f"task_{tt}"] = int(task_counts.get(tt, 0))

    # Notebook language counts — only count non-null values
    lang_counts = group["notebook_language"].dropna().value_counts()
    for lang in LANGUAGES:
        row[f"lang_{lang.lower()}"] = int(lang_counts.get(lang, 0))

    # Compute type counts — fillna so nulls don't wipe out counts
    compute_counts = group["cluster_type"].fillna("unknown").value_counts()
    for ct in COMPUTE_TYPES:
        row[f"compute_{ct}"] = int(compute_counts.get(ct, 0))

    rows.append(row)

summary = pd.DataFrame(rows)

# ── Write to Delta table ──────────────────────────────────────────────────────
print(f"Writing {len(summary)} rows to {full_summary}...")
spark_df = spark.createDataFrame(summary)
spark_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_summary)

print(f"Successfully wrote to {full_summary}")
summary.display()